# W261 Final Project Phase II - Flights EDA
**Team:** Section 3, Group 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np

spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
plt.rcParams.update({
    "figure.facecolor": "#F7F9FC",
    "axes.facecolor":   "#F7F9FC",
    "axes.edgecolor":   "#CCCCCC",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})
BLUE   = "#2166AC"
ORANGE = "#D6604D"
GREEN  = "#4DAC26"
GREY   = "#AAAAAA"

In [0]:
df = spark.read.parquet(
    "dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/"
)

In [0]:
n_rows = df.count()
n_cols = len(df.columns)
print("=" * 60)
print("FLIGHTS DATASET — 1-YEAR SAMPLE")
print("=" * 60)
print(f"  Rows    : {n_rows:,}")
print(f"  Columns : {n_cols}")
print("=" * 60)
 

In [0]:
data_dict = {
    # Temporal
    "YEAR":              ("int",    "Calendar year of the flight"),
    "QUARTER":           ("int",    "Quarter of the year (1–4)"),
    "MONTH":             ("int",    "Month of the year (1–12)"),
    "DAY_OF_MONTH":      ("int",    "Day of the month (1–31)"),
    "DAY_OF_WEEK":       ("int",    "Day of the week (1=Mon, 7=Sun)"),
    "FL_DATE":           ("string", "Full flight date (YYYY-MM-DD)"),
    # Carrier
    "OP_UNIQUE_CARRIER": ("string", "Unique carrier code (e.g., AA, DL)"),
    "OP_CARRIER":        ("string", "Reporting carrier code"),
    "TAIL_NUM":          ("string", "Aircraft tail number (registration)"),
    "OP_CARRIER_FL_NUM": ("int",    "Flight number assigned by carrier"),
    # Origin
    "ORIGIN":            ("string", "IATA code of the departure airport"),
    "ORIGIN_CITY_NAME":  ("string", "City name of the departure airport"),
    "ORIGIN_STATE_ABR":  ("string", "State abbreviation of departure airport"),
    # Destination
    "DEST":              ("string", "IATA code of the arrival airport"),
    "DEST_CITY_NAME":    ("string", "City name of the arrival airport"),
    "DEST_STATE_ABR":    ("string", "State abbreviation of arrival airport"),
    # Schedule & Timing
    "CRS_DEP_TIME":      ("int",    "Scheduled departure time (HHMM)"),
    "DEP_TIME":          ("int",    "Actual departure time (HHMM)"),
    "DEP_TIME_BLK":      ("string", "CRS departure time block (hourly bins)"),
    "CRS_ARR_TIME":      ("int",    "Scheduled arrival time (HHMM)"),
    "ARR_TIME":          ("int",    "Actual arrival time (HHMM)"),
    "CRS_ELAPSED_TIME":  ("double", "Scheduled elapsed flight time (minutes)"),
    # Route
    "DISTANCE":          ("double", "Great-circle distance between airports (miles)"),
    "DISTANCE_GROUP":    ("int",    "Distance interval (250-mile bins)"),
    # Delay — TARGET & related
    "DEP_DELAY":         ("double", "Departure delay in minutes (neg = early)"),
    "DEP_DEL15":         ("double", "TARGET: 1 if DEP_DELAY ≥ 15 min, else 0"),
    "DEP_DELAY_GROUP":   ("int",    "Delay group in 15-min intervals"),
    # Post-departure (EXCLUDE from features — data leakage)
    "ARR_DELAY":         ("double", "[LEAKAGE] Arrival delay in minutes"),
    "CARRIER_DELAY":     ("double", "[LEAKAGE] Delay due to carrier (min)"),
    "WEATHER_DELAY":     ("double", "[LEAKAGE] Delay due to weather (min)"),
    "NAS_DELAY":         ("double", "[LEAKAGE] Delay due to NAS (min)"),
    "LATE_AIRCRAFT_DELAY":("double","[LEAKAGE] Delay due to late aircraft (min)"),
    # Other
    "CANCELLED":         ("double", "1 if flight was cancelled"),
    "DIVERTED":          ("double", "1 if flight was diverted"),
}
 
print("\n── DATA DICTIONARY (relevant features) ──────────────────────────────")
print(f"{'Feature':<25} {'Type':<8} {'Description'}")
print("-" * 80)
for feat, (dtype, desc) in data_dict.items():
    tag = " ⚠ LEAKAGE" if "LEAKAGE" in desc else ""
    print(f"{feat:<25} {dtype:<8} {desc.replace('[LEAKAGE] ','')}{tag}")

In [0]:
print("\n── SCHEMA ───────────────────────────────────────────────────────────")
df.printSchema()

In [0]:
# Only check columns relevant to the modelling task
relevant_cols = [
    "YEAR", "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "FL_DATE",
    "OP_UNIQUE_CARRIER", "OP_CARRIER", "TAIL_NUM", "OP_CARRIER_FL_NUM",
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_ABR",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_TIME_BLK",
    "CRS_ARR_TIME", "ARR_TIME", "CRS_ELAPSED_TIME",
    "DISTANCE", "DISTANCE_GROUP",
    "DEP_DELAY", "DEP_DEL15", "DEP_DELAY_GROUP",
    "TAXI_OUT", "TAXI_IN", "AIR_TIME",
    "CANCELLED", "CANCELLATION_CODE", "DIVERTED",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]

null_exprs = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in relevant_cols]
null_counts_wide = df.select(null_exprs).toPandas().T
null_counts_wide.columns = ["null_count"]
null_counts_wide["null_pct"] = (null_counts_wide["null_count"] / n_rows * 100).round(2)
null_counts_wide = null_counts_wide.sort_values("null_pct", ascending=False)

# Only plot columns that actually have missing values
top_missing = null_counts_wide[null_counts_wide["null_pct"] > 0]

In [0]:
print(f"\n{'Column':<30} {'Null Count':>12} {'Null %':>8}")
print("-" * 55)
for col_name, row in null_counts_wide.iterrows():
    if row["null_pct"] > 0:
        print(f"{col_name:<30} {int(row['null_count']):>12,} {row['null_pct']:>7.1f}%")

In [0]:
top_missing = null_counts_wide[null_counts_wide["null_pct"] > 0].head(30)
 
fig, ax = plt.subplots(figsize=(12, max(4, len(top_missing) * 0.35)))
bars = ax.barh(top_missing.index[::-1], top_missing["null_pct"][::-1],
               color=[ORANGE if x > 50 else BLUE for x in top_missing["null_pct"][::-1]],
               edgecolor="white", linewidth=0.5)
ax.axvline(50, color=ORANGE, linestyle="--", linewidth=1, alpha=0.7, label="50% threshold")
ax.set_xlabel("Missing Values (%)")
ax.set_title("Missing Value Analysis — Flights Dataset (1-Year)\n"
             "Columns with > 0% missing shown; orange = > 50% missing", pad=12)
for bar, pct in zip(bars, top_missing["null_pct"][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/missing_values.png", dpi=150, bbox_inches="tight")
plt.show()
print("Columns with 0% missing are omitted from the chart above.")

In [0]:
num_cols = ["DEP_DELAY", "ARR_DELAY", "CRS_ELAPSED_TIME",
            "DISTANCE", "TAXI_OUT", "TAXI_IN", "AIR_TIME"]
 
print("\n── SUMMARY STATISTICS (numerical) ───────────────────────────────────")
display(df.select(num_cols).summary())
 


In [0]:
print("\n── TARGET VARIABLE: DEP_DEL15 ───────────────────────────────────────")

target_dist = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("DEP_DEL15")
      .count()
      .orderBy("DEP_DEL15")
      .toPandas()
)
target_dist["label"] = target_dist["DEP_DEL15"].map({0.0: "No Delay (<15 min)", 1.0: "Delay (≥15 min)"})
target_dist["pct"] = (target_dist["count"] / target_dist["count"].sum() * 100).round(1)

print(target_dist[["label", "count", "pct"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(target_dist["count"], labels=target_dist["label"], colors=[GREEN, ORANGE],
       autopct="%1.1f%%", startangle=90,
       wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("Target Variable Distribution — Class Imbalance\n(DEP_DEL15)", fontsize=13)

plt.tight_layout()
plt.savefig("/tmp/target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
delay_sample = (
    df.filter(F.col("DEP_DELAY").isNotNull())
      .select("DEP_DELAY")
      .sample(fraction=0.05, seed=42)
      .toPandas()
)

zoomed = delay_sample[delay_sample["DEP_DELAY"].between(-60, 180)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(zoomed["DEP_DELAY"], bins=80, color=BLUE, edgecolor="white", alpha=0.85)
ax.axvline(15, color=ORANGE, linestyle="--", linewidth=1.5, label="15-min threshold")
median_val = zoomed["DEP_DELAY"].median()
ax.axvline(median_val, color=GREEN, linestyle="-.",
           linewidth=1.5, label=f"Median = {median_val:.1f} min")
ax.set_title("Departure Delay Distribution")
ax.set_xlabel("Departure Delay (minutes)")
ax.set_ylabel("Number of Flights")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/delay_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# ── 9a. Delay rate by Month ───────────────────────────────────────────────────
delay_by_month = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("MONTH")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.count("*").alias("n_flights")
      )
      .orderBy("MONTH")
      .toPandas()
)
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
delay_by_month["month_name"] = delay_by_month["MONTH"].apply(lambda m: month_labels[m-1])
 
# ── 9b. Delay rate by Day of Week ─────────────────────────────────────────────
delay_by_dow = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("DAY_OF_WEEK")
      .agg(F.avg("DEP_DEL15").alias("delay_rate"), F.count("*").alias("n_flights"))
      .orderBy("DAY_OF_WEEK")
      .toPandas()
)
dow_labels = {1:"Mon",2:"Tue",3:"Wed",4:"Thu",5:"Fri",6:"Sat",7:"Sun"}
delay_by_dow["day_name"] = delay_by_dow["DAY_OF_WEEK"].map(dow_labels)
 
# ── 9c. Delay rate by Hour of Day ─────────────────────────────────────────────
delay_by_hour = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .withColumn("DEP_HOUR", (F.col("CRS_DEP_TIME") / 100).cast("int"))
      .groupBy("DEP_HOUR")
      .agg(F.avg("DEP_DEL15").alias("delay_rate"), F.count("*").alias("n_flights"))
      .orderBy("DEP_HOUR")
      .toPandas()
)
 
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
 
# Month
axes[0].bar(delay_by_month["month_name"], delay_by_month["delay_rate"] * 100,
            color=BLUE, edgecolor="white")
axes[0].set_title("Delay Rate by Month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Delay Rate (%)")
axes[0].set_xticklabels(delay_by_month["month_name"], rotation=45, ha="right")
for i, v in enumerate(delay_by_month["delay_rate"] * 100):
    axes[0].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=7)
 
# Day of Week
bar_colors_dow = [ORANGE if d == 5 else BLUE for d in delay_by_dow["DAY_OF_WEEK"]]
axes[1].bar(delay_by_dow["day_name"], delay_by_dow["delay_rate"] * 100,
            color=bar_colors_dow, edgecolor="white")
axes[1].set_title("Delay Rate by Day of Week\n(Friday highlighted)")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Delay Rate (%)")
for i, v in enumerate(delay_by_dow["delay_rate"] * 100):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=7)
 
# Hour of Day
axes[2].plot(delay_by_hour["DEP_HOUR"], delay_by_hour["delay_rate"] * 100,
             color=BLUE, linewidth=2, marker="o", markersize=4)
axes[2].fill_between(delay_by_hour["DEP_HOUR"], delay_by_hour["delay_rate"] * 100,
                     alpha=0.15, color=BLUE)
axes[2].set_title("Delay Rate by Scheduled Departure Hour")
axes[2].set_xlabel("Departure Hour (0–23)")
axes[2].set_ylabel("Delay Rate (%)")
axes[2].set_xticks(range(0, 24, 2))
axes[2].annotate("Late-day\ncongestion builds", xy=(18, delay_by_hour[delay_by_hour["DEP_HOUR"]==18]["delay_rate"].values[0]*100),
                 xytext=(14, 35), arrowprops=dict(arrowstyle="->", color=ORANGE), fontsize=8, color=ORANGE)
 
fig.suptitle("Temporal Patterns in Flight Delays", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/tmp/temporal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
carrier_stats = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("OP_UNIQUE_CARRIER")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.avg("DEP_DELAY").alias("avg_delay_min"),
          F.count("*").alias("n_flights")
      )
      .orderBy(F.desc("delay_rate"))
      .toPandas()
)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
# Delay rate
x = range(len(carrier_stats))
axes[0].bar(x, carrier_stats["delay_rate"] * 100,
            color=[ORANGE if v > carrier_stats["delay_rate"].mean() * 100 else BLUE
                   for v in carrier_stats["delay_rate"] * 100],
            edgecolor="white")
axes[0].axhline(carrier_stats["delay_rate"].mean() * 100, color=GREEN,
                linestyle="--", linewidth=1.5, label=f"Overall mean = {carrier_stats['delay_rate'].mean()*100:.1f}%")
axes[0].set_xticks(x)
axes[0].set_xticklabels(carrier_stats["OP_UNIQUE_CARRIER"], rotation=45, ha="right")
axes[0].set_title("Delay Rate by Carrier (≥15 min)")
axes[0].set_xlabel("Carrier Code")
axes[0].set_ylabel("Delay Rate (%)")
axes[0].legend(fontsize=9)
 
# Avg delay minutes vs flight volume (bubble chart)
sc = axes[1].scatter(carrier_stats["n_flights"], carrier_stats["avg_delay_min"],
                     s=carrier_stats["n_flights"] / carrier_stats["n_flights"].max() * 800,
                     c=carrier_stats["delay_rate"], cmap="RdYlGn_r",
                     alpha=0.8, edgecolors="white", linewidth=0.8)
for _, row in carrier_stats.iterrows():
    axes[1].annotate(row["OP_UNIQUE_CARRIER"],
                     (row["n_flights"], row["avg_delay_min"]),
                     textcoords="offset points", xytext=(4, 4), fontsize=7)
plt.colorbar(sc, ax=axes[1], label="Delay Rate")
axes[1].set_title("Avg Delay (min) vs. Flight Volume by Carrier\n(bubble size = flight volume)")
axes[1].set_xlabel("Number of Flights")
axes[1].set_ylabel("Average Departure Delay (minutes)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
 
fig.suptitle("Carrier-Level Delay Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/tmp/carrier_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
 
print("\n── CARRIER STATS TABLE ──────────────────────────────────────────────")
carrier_stats["delay_rate_pct"] = (carrier_stats["delay_rate"] * 100).round(1)
carrier_stats["avg_delay_min"]  = carrier_stats["avg_delay_min"].round(1)
carrier_stats["n_flights"]      = carrier_stats["n_flights"].apply(lambda x: f"{x:,}")
print(carrier_stats[["OP_UNIQUE_CARRIER","delay_rate_pct","avg_delay_min","n_flights"]]
      .rename(columns={"OP_UNIQUE_CARRIER":"Carrier",
                        "delay_rate_pct":"Delay Rate (%)",
                        "avg_delay_min":"Avg Delay (min)",
                        "n_flights":"# Flights"})
      .to_string(index=False))

In [0]:
origin_stats = (
    df.filter(F.col("DEP_DEL15").isNotNull())
      .groupBy("ORIGIN", "ORIGIN_CITY_NAME")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.count("*").alias("n_flights")
      )
      .filter(F.col("n_flights") >= 1000)   # min volume filter
      .orderBy(F.desc("delay_rate"))
      .toPandas()
)
 
top20 = origin_stats.head(20)
bot20 = origin_stats.tail(20)
 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
 
for ax, data, title, color in [
    (axes[0], top20, "Top 20 Airports — Highest Delay Rate", ORANGE),
    (axes[1], bot20.iloc[::-1], "Top 20 Airports — Lowest Delay Rate", GREEN)
]:
    ax.barh(data["ORIGIN"], data["delay_rate"] * 100, color=color, edgecolor="white")
    ax.set_xlabel("Delay Rate (%)")
    ax.set_title(title)
    for i, (rate, n) in enumerate(zip(data["delay_rate"] * 100, data["n_flights"])):
        ax.text(rate + 0.2, i, f"{rate:.1f}%  (n={n:,})", va="center", fontsize=7.5)
 
fig.suptitle("Origin Airport Delay Rates (airports with ≥ 1,000 flights)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("/tmp/airport_delay_rates.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
dist_delay = (
    df.filter(F.col("DEP_DEL15").isNotNull() & F.col("DISTANCE_GROUP").isNotNull())
      .groupBy("DISTANCE_GROUP")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.avg("DISTANCE").alias("avg_distance"),
          F.count("*").alias("n_flights")
      )
      .orderBy("DISTANCE_GROUP")
      .toPandas()
)
 
fig, ax = plt.subplots(figsize=(10, 4))
ax2 = ax.twinx()
 
bars = ax.bar(dist_delay["DISTANCE_GROUP"], dist_delay["delay_rate"] * 100,
              color=BLUE, alpha=0.7, edgecolor="white", label="Delay Rate (%)")
ax2.plot(dist_delay["DISTANCE_GROUP"], dist_delay["n_flights"],
         color=ORANGE, linewidth=2, marker="o", markersize=5, label="# Flights")
 
ax.set_xlabel("Distance Group (each group = 250 miles)")
ax.set_ylabel("Delay Rate (%)", color=BLUE)
ax2.set_ylabel("Number of Flights", color=ORANGE)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax.set_title("Delay Rate and Flight Volume by Distance Group\n"
             "(Group 1 = 0–250 mi, each subsequent group adds 250 mi)")
 
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/distance_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
corr_cols = [
    "DEP_DEL15", "MONTH", "DAY_OF_WEEK", "DISTANCE",
    "CRS_ELAPSED_TIME", "DISTANCE_GROUP"
]
 
corr_df = (
    df.select(corr_cols)
      .dropna()
      .sample(fraction=0.1, seed=42)
      .toPandas()
)
 
corr_matrix = corr_df.corr()
 
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor="white",
            ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix — Pre-Departure Numerical Features\n"
             "(target = DEP_DEL15; post-departure features excluded to prevent leakage)",
             pad=12)
plt.tight_layout()
plt.savefig("/tmp/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
 
# Text correlation with target
print("\n── CORRELATION WITH TARGET (DEP_DEL15) ─────────────────────────────")
target_corr = corr_matrix["DEP_DEL15"].drop("DEP_DEL15").sort_values(key=abs, ascending=False)
print(target_corr.to_string())

In [0]:
delay_causes = (
    df.filter(F.col("DEP_DEL15") == 1)
      .agg(
          F.avg("CARRIER_DELAY").alias("Carrier"),
          F.avg("WEATHER_DELAY").alias("Weather"),
          F.avg("NAS_DELAY").alias("NAS"),
          F.avg("SECURITY_DELAY").alias("Security"),
          F.avg("LATE_AIRCRAFT_DELAY").alias("Late Aircraft"),
      )
      .toPandas()
      .T.reset_index()
)
delay_causes.columns = ["Cause", "Avg_Minutes"]
delay_causes = delay_causes.sort_values("Avg_Minutes", ascending=False)
 
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(delay_causes["Cause"], delay_causes["Avg_Minutes"],
        color=[ORANGE, BLUE, GREEN, GREY, "#8B4513"], edgecolor="white")
ax.set_xlabel("Average Delay Contribution (minutes)")
ax.set_title("Average Delay Duration by Cause — Delayed Flights Only\n"
             "⚠ These features are POST-DEPARTURE and must NOT be used as model inputs")
for i, v in enumerate(delay_causes["Avg_Minutes"]):
    ax.text(v + 0.3, i, f"{v:.1f} min", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("/tmp/delay_causes.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
print("\n── CANCELLATIONS & DIVERSIONS ───────────────────────────────────────")
 
cancel_rate  = df.filter(F.col("CANCELLED") == 1).count() / n_rows * 100
divert_rate  = df.filter(F.col("DIVERTED")  == 1).count() / n_rows * 100
print(f"  Cancellation rate : {cancel_rate:.2f}%")
print(f"  Diversion rate    : {divert_rate:.2f}%")
 
cancel_by_code = (
    df.filter(F.col("CANCELLED") == 1)
      .groupBy("CANCELLATION_CODE")
      .count()
      .orderBy(F.desc("count"))
      .toPandas()
)
code_map = {"A": "Carrier", "B": "Weather", "C": "NAS", "D": "Security"}
cancel_by_code["Reason"] = cancel_by_code["CANCELLATION_CODE"].map(code_map)
print("\n  Cancellation breakdown:")
print(cancel_by_code[["CANCELLATION_CODE", "Reason", "count"]].to_string(index=False))
 

In [0]:
print("\n── TRAIN / VALIDATION / TEST SPLIT (time-based) ─────────────────────")
split_info = (
    df.groupBy("MONTH")
      .count()
      .orderBy("MONTH")
      .toPandas()
)
split_info["split"] = split_info["MONTH"].apply(
    lambda m: "Train" if m <= 8 else ("Validation" if m <= 10 else "Test")
)
split_info["pct"] = (split_info["count"] / split_info["count"].sum() * 100).round(1)
print(split_info[["MONTH","count","pct","split"]].to_string(index=False))
print("\n  Rationale: Time-based split prevents future data leaking into training.")
print("  Train: Jan–Aug (~67%), Validation: Sep–Oct (~17%), Test: Nov–Dec (~17%)")
 
 

In [0]:
print("\n── DUPLICATE CHECK ──────────────────────────────────────────────────")
n_dupes = df.count() - df.dropDuplicates().count()
print(f"  Duplicate rows: {n_dupes:,}")
if n_dupes == 0:
    print("  ✓ No duplicate rows found.")